In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight   # ← NEW
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df = pd.read_csv("imdb-movies-dataset.csv")
df.head()

,Poster,Title,Year,Certificate,Duration (min),Genre,Rating,Metascore,Director,Cast,Votes,Description,Review Count,Review Title,Review
0,https://m.media-amazon.com/images/M/MV5BYWRkZj...,The Idea of You,2023.0,R,115.0,"Comedy, Drama, Romance",6.4,67.0,Michael Showalter,"Anne Hathaway, Nicholas Galitzine, Ella Rubin,...","28,744","Solène, a 40-year-old single mom, begins an un...",166,Hypocrisy as an idea,"This film, as well as the reaction to it, is a..."
1,https://m.media-amazon.com/images/M/MV5BZGI4NT...,Kingdom of the Planet of the Apes,2023.0,PG-13,145.0,"Action, Adventure, Sci-Fi",7.3,66.0,Wes Ball,"Owen Teague, Freya Allan, Kevin Durand, Peter ...","22,248","Many years after the reign of Caesar, a young ...",183,A phenomenal start to another trilogy!,"I'm a big fan of all the planet of the apes, a..."
2,https://m.media-amazon.com/images/M/MV5BZjIyOT...,Unfrosted,2023.0,PG-13,97.0,"Biography, Comedy, History",5.5,42.0,Jerry Seinfeld,"Isaac Bae, Jerry Seinfeld, Chris Rickett, Rach...","18,401","In 1963 Michigan, business rivals Kellogg's an...",333,not funny,Pretty much the worst criticism you can lay on...
3,https://m.media-amazon.com/images/M/MV5BMjA5Zj...,The Fall Guy,2023.0,PG-13,126.0,"Action, Comedy, Drama",7.3,73.0,David Leitch,"Ryan Gosling, Emily Blunt, Aaron Taylor-Johnso...","38,953",A down-and-out stuntman must find the missing ...,384,Everything you needed and more!,Just got out of the Austin premier at SXSW and...
4,https://m.media-amazon.com/images/M/MV5BNTk1MT...,Challengers,2023.0,R,131.0,"Drama, Romance, Sport",7.7,82.0,Luca Guadagnino,"Zendaya, Mike Faist, Josh O'Connor, Darnell Ap...","32,517","Tashi, a former tennis prodigy turned coach, t...",194,"Watch ""Match Point"" instead",This is a tough one. I liked the concept and t...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Poster          10000 non-null  object 
 1   Title           10000 non-null  object 
 2   Year            9850 non-null   float64
 3   Certificate     7370 non-null   object 
 4   Duration (min)  9664 non-null   float64
 5   Genre           9993 non-null   object 
 6   Rating          9596 non-null   float64
 7   Metascore       7555 non-null   float64
 8   Director        9995 non-null   object 
 9   Cast            9961 non-null   object 
 10  Votes           9596 non-null   object 
 11  Description     10000 non-null  object 
 12  Review Count    9999 non-null   object 
 13  Review Title    9483 non-null   object 
 14  Review          9484 non-null   object 
dtypes: float64(4), object(11)
memory usage: 1.1+ MB


In [4]:
df.shape

(10000, 15)

In [5]:
df.columns

Index(['Poster', 'Title', 'Year', 'Certificate', 'Duration (min)', 'Genre',
       'Rating', 'Metascore', 'Director', 'Cast', 'Votes', 'Description',
       'Review Count', 'Review Title', 'Review'],
      dtype='object')

In [6]:
df.isnull().sum()

Poster               0
Title                0
Year               150
Certificate       2630
Duration (min)     336
Genre                7
Rating             404
Metascore         2445
Director             5
Cast                39
Votes              404
Description          0
Review Count         1
Review Title       517
Review             516
dtype: int64

In [7]:
df.dropna(inplace=True)

In [8]:
df.isnull().sum()

Poster            0
Title             0
Year              0
Certificate       0
Duration (min)    0
Genre             0
Rating            0
Metascore         0
Director          0
Cast              0
Votes             0
Description       0
Review Count      0
Review Title      0
Review            0
dtype: int64

In [9]:
df['Sentiment'] = df['Rating'].apply(lambda x: 1 if x >= 7 else 0)

df.head()

,Poster,Title,Year,Certificate,Duration (min),Genre,Rating,Metascore,Director,Cast,Votes,Description,Review Count,Review Title,Review,Sentiment
0,https://m.media-amazon.com/images/M/MV5BYWRkZj...,The Idea of You,2023.0,R,115.0,"Comedy, Drama, Romance",6.4,67.0,Michael Showalter,"Anne Hathaway, Nicholas Galitzine, Ella Rubin,...","28,744","Solène, a 40-year-old single mom, begins an un...",166,Hypocrisy as an idea,"This film, as well as the reaction to it, is a...",0
1,https://m.media-amazon.com/images/M/MV5BZGI4NT...,Kingdom of the Planet of the Apes,2023.0,PG-13,145.0,"Action, Adventure, Sci-Fi",7.3,66.0,Wes Ball,"Owen Teague, Freya Allan, Kevin Durand, Peter ...","22,248","Many years after the reign of Caesar, a young ...",183,A phenomenal start to another trilogy!,"I'm a big fan of all the planet of the apes, a...",1
2,https://m.media-amazon.com/images/M/MV5BZjIyOT...,Unfrosted,2023.0,PG-13,97.0,"Biography, Comedy, History",5.5,42.0,Jerry Seinfeld,"Isaac Bae, Jerry Seinfeld, Chris Rickett, Rach...","18,401","In 1963 Michigan, business rivals Kellogg's an...",333,not funny,Pretty much the worst criticism you can lay on...,0
3,https://m.media-amazon.com/images/M/MV5BMjA5Zj...,The Fall Guy,2023.0,PG-13,126.0,"Action, Comedy, Drama",7.3,73.0,David Leitch,"Ryan Gosling, Emily Blunt, Aaron Taylor-Johnso...","38,953",A down-and-out stuntman must find the missing ...,384,Everything you needed and more!,Just got out of the Austin premier at SXSW and...,1
4,https://m.media-amazon.com/images/M/MV5BNTk1MT...,Challengers,2023.0,R,131.0,"Drama, Romance, Sport",7.7,82.0,Luca Guadagnino,"Zendaya, Mike Faist, Josh O'Connor, Darnell Ap...","32,517","Tashi, a former tennis prodigy turned coach, t...",194,"Watch ""Match Point"" instead",This is a tough one. I liked the concept and t...,1


In [10]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [11]:
df['full_review'] = df['Review Title'].apply(clean_text) + ' ' + df['Review'].apply(clean_text)

In [12]:
X = df['full_review']
y = df['Sentiment']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [14]:
max_words = 10000
max_len = 300

tokenizer = Tokenizer(num_words=max_words,oov_token="<OOV>")

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len,padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len,padding='post')

In [15]:
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = {0: cw[0], 1: cw[1]}
print("Class weights:", class_weight)

Class weights: {0: np.float64(0.7696814903846154), 1: np.float64(1.4270194986072424)}


In [16]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [17]:
rnn_model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    SimpleRNN(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
rnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

C:\Users\patel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
rnn_history = rnn_model.fit(
    X_train_pad, y_train,
    epochs=10,                      
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],         
    class_weight=class_weight       
)

Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 5s 108ms/step - accuracy: 0.5478 - loss: 0.7027 - val_accuracy: 0.6000 - val_loss: 0.6629
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.6449 - loss: 0.6458 - val_accuracy: 0.6341 - val_loss: 0.6673
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.7338 - loss: 0.5607 - val_accuracy: 0.6224 - val_loss: 0.6840


In [19]:
rnn_pred = (rnn_model.predict(X_test_pad) > 0.5).astype("int32")
rnn_acc  = accuracy_score(y_test, rnn_pred)
print("RNN Accuracy:", rnn_acc)
print(classification_report(y_test, rnn_pred))

41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
RNN Accuracy: 0.6104605776736924
              precision    recall  f1-score   support

           0       0.70      0.70      0.70       832
           1       0.44      0.44      0.44       449

    accuracy                           0.61      1281
   macro avg       0.57      0.57      0.57      1281
weighted avg       0.61      0.61      0.61      1281



In [ ]:
lstm_model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(LSTM(64)),        
    Dropout(0.3),
    Dense(64, activation='relu'),   
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

C:\Users\patel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
lstm_history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],         
    class_weight=class_weight       
)

Epoch 1/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 20s 262ms/step - accuracy: 0.5859 - loss: 0.6830 - val_accuracy: 0.6332 - val_loss: 0.6362
Epoch 2/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 14s 207ms/step - accuracy: 0.6098 - loss: 0.6369 - val_accuracy: 0.6234 - val_loss: 0.6280
Epoch 3/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 14s 208ms/step - accuracy: 0.7596 - loss: 0.4893 - val_accuracy: 0.6244 - val_loss: 0.6981
Epoch 4/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 13s 199ms/step - accuracy: 0.8612 - loss: 0.3336 - val_accuracy: 0.6556 - val_loss: 0.8109


In [22]:
lstm_pred = (lstm_model.predict(X_test_pad) > 0.5).astype("int32")
lstm_acc  = accuracy_score(y_test, lstm_pred)
print("LSTM Accuracy:", lstm_acc)
print(classification_report(y_test, lstm_pred))

41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step
LSTM Accuracy: 0.5940671350507416
              precision    recall  f1-score   support

           0       0.78      0.53      0.63       832
           1       0.45      0.72      0.55       449

    accuracy                           0.59      1281
   macro avg       0.61      0.62      0.59      1281
weighted avg       0.66      0.59      0.60      1281



In [23]:
gru_model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    GRU(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

C:\Users\patel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
gru_history = gru_model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],         
    class_weight=class_weight       
)

Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 250ms/step - accuracy: 0.5913 - loss: 0.6872 - val_accuracy: 0.5990 - val_loss: 0.6639
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 229ms/step - accuracy: 0.5966 - loss: 0.6772 - val_accuracy: 0.6020 - val_loss: 0.6874


In [25]:
gru_pred = (gru_model.predict(X_test_pad) > 0.5).astype("int32")
gru_acc  = accuracy_score(y_test, gru_pred)
print("GRU Accuracy:", gru_acc)
print(classification_report(y_test, gru_pred))

41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
GRU Accuracy: 0.5979703356752537
              precision    recall  f1-score   support

           0       0.70      0.67      0.68       832
           1       0.43      0.47      0.45       449

    accuracy                           0.60      1281
   macro avg       0.57      0.57      0.57      1281
weighted avg       0.61      0.60      0.60      1281



In [26]:
results = pd.DataFrame({
    'Model':    ['RNN', 'LSTM', 'GRU'],
    'Accuracy': [rnn_acc, lstm_acc, gru_acc]
})
print(results)

best_model = results.sort_values('Accuracy', ascending=False) 
print("\nBest model is:")
print(best_model.iloc[0])

  Model  Accuracy
0   RNN  0.610461
1  LSTM  0.594067
2   GRU  0.597970

Best model is:
Model            RNN
Accuracy    0.610461
Name: 0, dtype: object


In [27]:
print("Best model is:")
print(best_model.iloc[0])

Best model is:
Model            RNN
Accuracy    0.610461
Name: 0, dtype: object
